<a href="https://colab.research.google.com/github/maclandrol/cours-ia-med/blob/master/devoirs/Homework_Chest_XRay_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Devoir 2: Estimation d'Âge sur Radiographies Thoraciques

**Enseignant:** Emmanuel Noutahi, PhD  
**Note:** /20 points

---

## Objectif

Peut-on estimer l'âge d'un patient à partir d'une radiographie thoracique ? Ce devoir explore cette question avec un modèle pré-entraîné et évalue la qualité des prédictions.

## Barème (/20)

1. Setup (3 pts)
2. Prédictions (6 pts)
3. Évaluation (7 pts)
4. Réflexion (4 pts)

**Ressources:**
- Doc: https://mlmed.org/torchxrayvision/
- Dataset: https://huggingface.co/datasets/BahaaEldin0/NIH-Chest-Xray-14

## Instructions

- Travail individuel, en binôme ou en trinôme (max 3 personnes)
- Les sections à compléter sont marquées `# TODO`
- **N'indiquez pas la solution** : complétez le code et rédigez les réponses vous-même
- Exécutez les cellules dans l'ordre
- Avant remise : **Tout exécuter**, vérifier les erreurs, mettre vos noms

### ÉQUIPE

- **<font color='red'>Nom Prénoms</font>**

## Partie 1: Setup (3 pts)

In [ ]:
!pip install -q torchxrayvision torch torchvision datasets matplotlib pandas scikit-learn seaborn

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torchxrayvision as xrv
from datasets import load_dataset
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

### Question 1.1 (1 pt): Contexte clinique

Les radiographies thoraciques contiennent-elles de l'information pour prédire l'âge d'un patient ? Argumentez en quelques lignes (aspects anatomiques, physiologiques, etc.).

**Votre réponse:**

### Question 1.2 (1 pts): Charger le modèle

Chargez le **Riken AgeModel** (MAE ≈ 3.7 ans sur NIH). Consultez la doc torchxrayvision pour trouver la bonne classe.

**À faire:** Complétez la ligne `model = ...`

In [ ]:
# TODO: Chargez le model Riken AgeModel
# Indice: utilisez la resource donnée
# https://mlmed.org/torchxrayvision/models.html 
model = ...

model = model.to(device)
model.eval()
print(f"Modèle chargé. Cible: {model.targets}")

### Question 1.3 (1 pt): Charger le dataset

Chargez le dataset NIH Chest X-ray 14 (split `test`), puis sous-échantillonnez **1000 images** avec une graine pour la reproductibilité.

**À faire:** Complétez les lignes marquées TODO.

In [ ]:
from datasets import load_dataset, Dataset

print("Chargement du dataset...")
base = "https://huggingface.co/datasets/BahaaEldin0/NIH-Chest-Xray-14/resolve/main"

data_files = {
    "test": [
        f"{base}/data/test-00000-of-00010.parquet",
        f"{base}/data/test-00001-of-00010.parquet",
        f"{base}/data/test-00002-of-00010.parquet",
    ]
}

stream = load_dataset(
    "parquet",
    data_files=data_files,
    split="test",
    streaming=True,
)

# ==========
# TODO: Sous-échantillonnez 1000 images
N_IMAGES = ...
dataset_sample = Dataset.from_list(list(stream.take(N_IMAGES)))
# ========

print(f"Échantillon: {len(dataset_sample)} images")
print(f"Colonnes: {dataset_sample.column_names}")

## Partie 2: Prédictions (6 pts)

### Question 2.1 (3 pts): Fonction de prédiction

Implémentez une fonction `predict_age(image, model)` qui :
1. Convertit l'image PIL en numpy, normalise avec `xrv.datasets.normalize(img, 255)`
2. Passe en grayscale si besoin (un seul canal)
3. Ajoute les dimensions batch et canal pour PyTorch (forme attendue : `(1, 1, H, W)`)
4. Convertit en tensor, envoie sur `device`, appelle le modèle, retourne l'âge prédit (float)

**Indices:** `torch.no_grad()`, `torch.from_numpy()`, `model(x)`.

In [1]:
def predict_age(image, model):
    """Prédit l'âge depuis une image PIL."""
    with torch.no_grad():
        img = np.array(image)
        # TODO: Normalisez l'image sur les bonnes valeurs.
        img = ...

        if len(img.shape) > 2:
            img = img[:, :, 0]
        img = img[None, None, :, :].astype(np.float32)
        x = torch.from_numpy(img.astype(np.float32)).to(device)
        
        # TODO: prediction du model
        pred = ...
        return float(pred[0, 0].cpu())

### Question 2.2 (1 pts): Tester sur un exemple

Récupérez la première image du dataset, son âge réel (colonne `Patient Age`), et affichez l'image avec âge réel et prédit.

**À faire:** Complétez les variables `sample`, `img`, `true_age`, `pred_age`.

In [ ]:
# TODO: Récupérez dataset_sample[0], l'image, l'âge réel (colonne 'Patient Age')
sample = ...
img = ...
true_age = ...
pred_age = predict_age(img, model)

plt.figure(figsize=(5, 5))
plt.imshow(img, cmap='gray')
plt.title(f"Âge réel: {true_age} ans | Prédit: {pred_age:.1f} ans")
plt.axis('off')
plt.tight_layout()
plt.show()

### Question 2.3 (2 pts): Analyser au moins 100 images

Bouclez sur les **100 premières** images (ou plus), prédisez l'âge pour chacune, et construisez un DataFrame `results_df` avec colonnes : `age_reel`, `age_predit`, `erreur quadratique`.

La formule de l'erreur quadratique est:

$\text{MSE}_i = (y_i - \hat{y}_i)^2$


**À faire:** Complétez la boucle et la création du DataFrame.

In [ ]:
N_SAMPLES = 200  # au moins 100 pour des stats stables
AGE_COL = 'Patient Age'

rows = []
for i in range(N_SAMPLES):
    item = dataset_sample[i]
    # TODO: Récupérez true (âge réel) et pred (âge prédit via predict_age)
    y_true = ...
    y_pred = ...
    rows.append({"age_reel": y_true, "age_predit": y_pred})

results_df = pd.DataFrame(rows)
# TODO: Ajoutez les colonnes 'erreur' pour l'erreur quadratique (MSE)

results_df['erreur'] = ...
print(results_df.head(10))

## Partie 3: Évaluation (7 pts)

### Question 3.1 (1 pts): Métriques (MAE, MSE, R²)

Calculez MAE, RMSE (racine de MSE), MSE et R². Utilisez les fonctions `mean_absolute_error`, `root_mean_squared_error` et `r2_score` de sklearn qui sont déjà importées et définies. Stockez les résultats dans un dictionnaire `metrics` et affichez-les.

**À faire:** Complétez le calcul des métriques.

In [ ]:
# TODO: Calculez mae, mse, rmse, r2
mae = ...
rmse = ...  # racine carrée de mse
r2 = ...

print("MÉTRIQUES")
print("-" * 40)
print(f"  MAE: {mae:.3f}")
print(f"  RMSE: {rmse:.3f}")
print(f"  R²: {r2:.3f}")
print("-" * 40)
print(f"Référence paper: MAE ≈ 3.67 ans")

### Question 3.2 (2 pts): Visualisations

Créez 3 graphiques côte à côte :
1. **Scatter** : âge réel vs âge prédit, avec la droite "parfaite" (y=x)
2. **Histogramme** : distribution des erreurs (prédit - réel)
3. **Scatter** : âge réel vs erreur absolue, avec une ligne horizontale au niveau du MAE

**À faire:** Complétez les appels `ax1.scatter`, `ax2.hist`, `ax3.scatter` et les options de titre/labels.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
ax1, ax2, ax3 = axes

# TODO: Scatter âge réel vs prédit + droite y=x
ax1.scatter(..., ..., alpha=0.6, s=50)
lo, hi = results_df['age_reel'].min(), results_df['age_reel'].max()
ax1.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Parfait')
ax1.set_xlabel('Âge réel (ans)')
ax1.set_ylabel('Âge prédit (ans)')
ax1.set_title(f'Réel vs prédit (MAE={mae:.2f})')
ax1.legend()
ax1.grid(alpha=0.3)

# TODO: Histogramme des erreurs
ax2.hist(..., bins=20, edgecolor='black', alpha=0.7)
ax2.axvline(0, color='red', ls='--', lw=1.5)
ax2.set_xlabel(...)
ax2.set_ylabel(...)
ax2.set_title('Distribution des erreurs')
ax2.grid(alpha=0.3)

# TODO: Erreur absolue vs âge réel + ligne MAE
ax3.scatter(..., ..., alpha=0.6, s=50)
ax3.axhline(mae, color='green', ls='--', lw=1.5, label=f'MAE={mae:.1f}')
ax3.set_xlabel('Âge réel (ans)')
ax3.set_ylabel('Erreur absolue (ans)')
ax3.set_title('Erreur abs vs âge')
ax3.legend()
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Question 3.3 (1 pt): Analyse par groupe d'âge

Créez une fonction `categorize_age(age)` qui retourne `<30`, `30-50`, `50-70`, ou `>70`. Ajoutez une colonne `groupe_age` à `results_df` et affichez le MAE par groupe.

In [ ]:
def categorize_age(age):
    # TODO: Retourner '<30', '30-50', '50-70', ou '>70' en fonction de l'age
    ...

results_df['groupe_age'] = results_df['age_reel'].apply(categorize_age)

print("MAE par groupe d'âge:")
for g in ['<30', '30-50', '50-70', '>70']:
    s = results_df[results_df['groupe_age'] == g]
    if len(s) > 0:
        print(f"  {g}: {s['erreur'].mean():.2f} ans (n={len(s)})")

### Question 3.4 (1pt): Analyse des performance selon les autres variable non-utilisées

Le modèle ne reçoit que l'image. Comparez l'erreur selon le sexe  et la vue de la prise pour chercher d'éventuels biais.

In [ ]:
df = results_df.copy()
variable_sex = ...
variable_vue = ...
df['sexe'] = [dataset_sample[i][variable_sex] for i in range(N_SAMPLES)]
df['vue'] = [dataset_sample[i][variable_vue] for i in range(N_SAMPLES)]

print("MSE par sexe:", df.groupby('sexe')['erreur'].mean().to_dict())
print("MSE par vue:", df.groupby('vue')['erreur'].mean().to_dict())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
df.boxplot(column='erreur', by='sexe', ax=ax1)
ax1.set_title('Erreur par sexe')
plt.suptitle('')
df.boxplot(column='erreur', by='vue', ax=ax2)
ax2.set_title('Erreur par vue')
plt.suptitle('')
plt.tight_layout()
plt.show()

### Question 3.5 (2pts):  Visualisation de quelques prédictions 

Affichez une grille 3×3 d'images avec âge réel et prédit en titre. Vous devez afficher les 
prédictions pour les 9 derniers examples dans `dataset_sample`. Inspirez vous de l'example précédent dans le notebook.

In [ ]:
n_show = 9
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for k in range(n_show):
    ax = axes.flat[k]
    # TODO: completez pour afficher l'image et trouver la valeur de y_true et y_pred
    # RAPPEL: vous devez utiliser les neuf derniers échantillons
    ...
    
    ax.set_title(f"Réel: {y_true} | Prédit: {y_pred:.1f}", fontsize=9)
    ax.axis('off')
plt.suptitle("Exemples de prédictions", fontsize=12)
plt.tight_layout()
plt.show()

## Partie 4: Réflexion (4 pts)

Points à retenir pour la réflexion :
- **MAE / RMSE** : écart typique entre âge prédit et réel
- **R²** : qualité de la corrélation entre les prédictions et l'âge réel
- **Erreur vs âge** : homogénéité du modèle sur les tranches d'âge ?
- **Erreur vs sexe / vue, etc** : signes de biais ?

### Question 4.1 (2 pt): Limites 

1. Lister les cas cliniques ou ce modèle pourrait être utilisé. 
2. Avec l'erreur de prédiction que vous avez observé, le modèle est-il suffisant pour ces usages cliniques ?

**Votre réponse:**

### Question 4.2 (2 pt): Biais

1. Le modèle est entraîné sur des patients américains (NIH). Quels biais potentiels avez-vous identifié basé sur les figures de la **section 3.**. Quels autres biais pourrait exister si on utilisait le modèle sur des patients africains ? (3–5 lignes)
2. Quelles sont les autres limitations potentielles de ce modèle dans les contextes d'utilisations que vous avez identifiés plus haut. 

**Votre réponse:**

## BONUS (max 2pts): Prédiction sur vos propre images

Sur Colab : uploadez une radiographie et prédisez l'âge. Que pensez vous de cet outil ?

In [ ]:
try:
    from google.colab import files
    import io
    uploaded = files.upload()
    if uploaded:
        f = list(uploaded.keys())[0]
        img = ...
        age = predict_age(img, model)
        plt.imshow(img, cmap='gray')
        plt.title(f"Âge prédit: {age:.1f} ans")
        plt.axis('off')
        plt.show()
        print(f"IC approx: [{age-3.7:.1f}, {age+3.7:.1f}] ans")
except ImportError:
    print("Bonus Colab uniquement.")

## Remise

1. Exécution → Tout exécuter
2. Vérifier les erreurs
3. Compléter les réponses textuelles
4. Sauvegarder et partager le lien

**Bon travail !**